# Model 4 — LoRA fine-tuned ModernBERT (GPU recommended)

Predicts the **6-vector of compression perplexities** from prompt text.

- Data pulled live from GitHub (`perplexity_wide_complete.csv`).
- Targets in **log space**; inverted to raw perplexity for reporting.
- **Stratified random 70/30 split by dataset**, `random_state=42`.
- Trained model saved to `artifacts/`.

Requires `kv_common.py` in the same folder.

> **GPU strongly recommended.** On CPU this is very slow. Colab/Kaggle free GPU works.

## Colab setup

Run this cell **first**. It clones the repo (so `kv_common.py` is available),
installs packages, and optionally mounts Google Drive so your cached embeddings
and trained models survive a disconnect.

> For notebook 04 (LoRA), also enable a GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# Clone the repo so kv_common.py and artifacts live in one place.
import os
if not os.path.exists("KVCacheCompression"):
    !git clone -q https://github.com/yoshikodes/KVCacheCompression.git
# Work inside the notebooks folder (edit if your notebooks live elsewhere).
if os.path.basename(os.getcwd()) != "notebooks":
    %cd KVCacheCompression/notebooks
print("cwd:", os.getcwd())
assert os.path.exists("kv_common.py"), "kv_common.py not found"


In [ ]:
# Install packages not preinstalled on Colab.
# If you get an import error right after this, do Runtime -> Restart, then re-run from the top.
!pip install -q transformers peft accelerate scikit-learn scipy requests

In [ ]:
# Confirm GPU is active (needed for reasonable LoRA training speed).
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Enable via Runtime -> Change runtime type -> T4 GPU, then restart.")

In [ ]:
# Optional but recommended: mount Drive so artifacts/ persists across sessions.
# Without this, trained models and cached embeddings are lost on disconnect,
# and notebook 05 will only see models trained in the current session.
USE_DRIVE = True  # set False to keep everything ephemeral

import os
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ART_DIR = "/content/drive/MyDrive/kv_artifacts"
        os.makedirs(ART_DIR, exist_ok=True)
        # Link ./artifacts -> Drive so all the notebook's save paths persist.
        if os.path.islink("artifacts") or os.path.exists("artifacts"):
            if not os.path.islink("artifacts"):
                import shutil; shutil.rmtree("artifacts", ignore_errors=True)
        if not os.path.exists("artifacts"):
            os.symlink(ART_DIR, "artifacts")
        print("artifacts -> ", os.path.realpath("artifacts"))
    except Exception as e:
        print("Drive mount skipped:", e)
        os.makedirs("artifacts", exist_ok=True)
else:
    os.makedirs("artifacts", exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, os, joblib
import kv_common as kv

os.makedirs("artifacts", exist_ok=True)
df = kv.load_data()
train_df, test_df = kv.stratified_split(df)

LOG_SPACE = True
y_train = kv.get_targets(train_df, log_space=LOG_SPACE)
y_test  = kv.get_targets(test_df,  log_space=LOG_SPACE)
baseline = kv.baseline_predict_mean(y_train, len(y_test))
baseline_metrics = kv.evaluate(y_test, baseline)
print("Mean-baseline OVERALL MAE_log:",
      round(baseline_metrics[baseline_metrics.setting=='OVERALL'].MAE_log.iloc[0], 4))


## Tokenize

In [ ]:
import torch
from transformers import AutoTokenizer
MODEL_NAME = "answerdotai/ModernBERT-base"
MAX_LEN = 512
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tok(list(texts), truncation=True, max_length=MAX_LEN,
               padding="max_length", return_tensors="pt")
enc_train = encode(train_df["prompt"]); enc_test = encode(test_df["prompt"])

# Standardize targets.
y_mean = y_train.mean(0); y_std = y_train.std(0)+1e-8
Ytr = torch.tensor((y_train-y_mean)/y_std, dtype=torch.float32)

## Attach LoRA + a 6-output regression head

In [ ]:
import torch.nn as nn
from transformers import AutoModel
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

class LoRARegressor(nn.Module):
    def __init__(self, model_name, n_out):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.1,
                         target_modules=["Wqkv","Wo"], bias="none")
        self.backbone = get_peft_model(self.backbone, cfg)
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(h,256), nn.GELU(),
                                  nn.Dropout(0.1), nn.Linear(256, n_out))
    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:,0]   # CLS
        return self.head(pooled)

model = LoRARegressor(MODEL_NAME, len(kv.SETTINGS)).to(device)
model.backbone.print_trainable_parameters()

## Train

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
ds = TensorDataset(enc_train["input_ids"], enc_train["attention_mask"], Ytr)
dl = DataLoader(ds, batch_size=16, shuffle=True)
opt = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 2e-4},
    {"params": model.head.parameters(), "lr": 1e-3}], weight_decay=1e-4)

EPOCHS = 8
for ep in range(EPOCHS):
    model.train(); tot=0
    for ids, am, yb in dl:
        ids, am, yb = ids.to(device), am.to(device), yb.to(device)
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(ids, am), yb)
        loss.backward(); opt.step(); tot += loss.item()
    print(f"epoch {ep+1}/{EPOCHS}  mean loss {tot/len(dl):.4f}")

## Evaluate

In [ ]:
model.eval()
preds=[]
with torch.no_grad():
    for i in range(0, len(test_df), 32):
        ids = enc_test["input_ids"][i:i+32].to(device)
        am  = enc_test["attention_mask"][i:i+32].to(device)
        preds.append(model(ids, am).cpu().numpy())
import numpy as np
pred = np.vstack(preds) * y_std + y_mean
metrics_lora = kv.evaluate(y_test, pred)
print(metrics_lora.round(4).to_string(index=False))
kv.print_comparison(metrics_lora, baseline_metrics)

## Save (LoRA adapter + head)

In [ ]:
import os
os.makedirs("artifacts/model4_lora", exist_ok=True)
model.backbone.save_pretrained("artifacts/model4_lora/adapter")
torch.save({"head": model.head.state_dict(), "y_mean": y_mean, "y_std": y_std,
            "model_name": MODEL_NAME, "settings": kv.SETTINGS,
            "log_space": LOG_SPACE}, "artifacts/model4_lora/head.pt")
metrics_lora.to_csv("artifacts/model4_lora_metrics.csv", index=False)
print("saved artifacts/model4_lora/")